In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from scipy.stats import uniform, randint
from tensorflow.keras import regularizers

In [4]:
df = pd.read_parquet('merge/result/Segment_merge_ver_02.parquet')

df

,기준년월,ID,Segment,입회일자_신용,수신거부여부_TM,이용카드수_신용체크,최종유효년월_신용_이용가능,이용여부_3M_해외겸용_본인,카드이용한도금액,CA이자율_할인전,...,혜택수혜금액_R3M,월중평잔,평잔_일시불_6M,인입일수_ARS_R6M,방문후경과월_앱_R6M,불만제기후경과월_R12M,컨택건수_이용유도_TM_R6M,컨택건수_이용유도_EM_R6M,잔액_신판ca최대한도소진율_r6m,변동률_RV일시불평잔
0,201807,TRAIN_000000,D,20130101,0,1,202110.0,0,19354,22.995207,...,3,17237,2440,8,6,12,3,57,0.849842,0.999998
1,201807,TRAIN_000001,E,20170801,0,1,202112.0,0,9996,14.793821,...,0,7967,2677,0,6,12,2,2,0.851009,1.092698
2,201807,TRAIN_000002,C,20080401,0,1,202111.0,0,88193,22.014276,...,121,59917,9118,1,0,12,2,12,0.938161,1.006124
3,201807,TRAIN_000003,D,20160501,0,1,202201.0,1,19062,22.998014,...,3,27854,884,10,6,12,2,35,1.135424,0.999998
4,201807,TRAIN_000004,E,20180601,0,1,202201.0,1,177222,14.661948,...,0,0,21,0,6,0,7,0,0.000000,0.999998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,201812,TRAIN_399995,E,20010701,0,1,202110.0,1,20070,15.243670,...,0,0,0,0,6,0,0,0,0.032439,0.999998
2399996,201812,TRAIN_399996,D,20170701,0,1,202110.0,1,84217,14.843464,...,164,29429,12524,0,6,12,0,58,0.168081,0.999998
2399997,201812,TRAIN_399997,C,20090501,1,1,202110.0,1,52612,17.038599,...,0,7383,3241,0,6,12,0,0,0.190393,0.999998
2399998,201812,TRAIN_399998,E,20130101,1,0,202202.0,0,10002,15.182880,...,0,0,0,0,6,0,0,0,0.012677,0.999998


In [3]:
# ID, Segment 분리
X = df.drop(columns=['ID', 'Segment'])
y = df['Segment']

# Label Encoding (Segment가 문자일 경우)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 데이터 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 학습/검증 분리
X_train, X_valid, y_train, y_valid = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [4]:
# 모델 생성 함수
def create_model(hidden_units=128, dropout_rate=0.4, learning_rate=0.001):
    model = Sequential()
    
    # 첫 번째 은닉층: L2 정규화 추가
    model.add(Dense(hidden_units,
                    input_shape=(X_train.shape[1],),
                    activation='relu',
                    kernel_regularizer=regularizers.l2(0.001)))
    model.add(Dropout(dropout_rate))

    # 두 번째 은닉층: L2 정규화 추가
    model.add(Dense(hidden_units // 2,
                    activation='relu',
                    kernel_regularizer=regularizers.l2(0.001)))
    model.add(Dropout(dropout_rate))

    # 출력층 (다중 분류용)
    model.add(Dense(len(np.unique(y_encoded)), activation='softmax'))

    # 컴파일
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [5]:
# KerasClassifier로 감싸기
model = KerasClassifier(build_fn=create_model, verbose=0)

C:\Users\Lee\AppData\Local\Temp\ipykernel_11392\343470949.py:2: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  model = KerasClassifier(build_fn=create_model, verbose=0)


In [6]:
# 파라미터 튜닝 범위 정의
param_dist = {
    'hidden_units': [64, 128, 256],
    'dropout_rate': uniform(0.2, 0.3),
    'learning_rate': uniform(0.0005, 0.005),
    'epochs': [10, 20],
    'batch_size': [64, 128]
}

# RandomizedSearchCV 설정
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=5,
    cv=2,
    error_score='raise',
    verbose=2,  
    random_state=42,
    n_jobs=1
)

In [7]:
# 학습
search.fit(X_train, y_train)

Fitting 2 folds for each of 5 candidates, totalling 10 fits
[CV] END batch_size=64, dropout_rate=0.4389628960580698, epochs=10, hidden_units=256, learning_rate=0.0043984550013638464; total time= 4.0min
[CV] END batch_size=64, dropout_rate=0.4389628960580698, epochs=10, hidden_units=256, learning_rate=0.0043984550013638464; total time= 4.0min
[CV] END batch_size=64, dropout_rate=0.24680559213273096, epochs=10, hidden_units=256, learning_rate=0.0007904180608409974; total time= 4.0min
[CV] END batch_size=64, dropout_rate=0.24680559213273096, epochs=10, hidden_units=256, learning_rate=0.0007904180608409974; total time= 4.1min
[CV] END batch_size=128, dropout_rate=0.30011258334170654, epochs=20, hidden_units=256, learning_rate=0.0006029224714790123; total time= 3.8min
[CV] END batch_size=128, dropout_rate=0.30011258334170654, epochs=20, hidden_units=256, learning_rate=0.0006029224714790123; total time= 3.7min
[CV] END batch_size=128, dropout_rate=0.41659963168004743, epochs=20, hidden_units

,estimator,<keras.wrappe...001EF05553EE0>
,param_distributions,"{'batch_size': [64, 128], 'dropout_rate': <scipy.stats....001EF055529E0>, 'epochs': [10, 20], 'hidden_units': [64, 128, ...], ...}"
,n_iter,5
,scoring,None
,n_jobs,1
,refit,True
,cv,2
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,'raise'


In [8]:
# 검증 성능 확인
y_pred = search.best_estimator_.predict(X_valid)
acc = accuracy_score(y_valid, y_pred)
print(f"Validation Accuracy: {acc:.4f}")
print("Best Parameters:", search.best_params_)

15000/15000 [==============================] - 11s 718us/step
Validation Accuracy: 0.8753
Best Parameters: {'batch_size': 128, 'dropout_rate': 0.30011258334170654, 'epochs': 20, 'hidden_units': 256, 'learning_rate': 0.0006029224714790123}


In [9]:
# 테스트 데이터 전처리
test = pd.read_parquet('merge/result/Segment_merge_test_ver_02.parquet')
X_test = test.drop(columns=['ID'])
X_test_scaled = scaler.transform(X_test)

In [10]:
# 예측
y_test_pred = search.best_estimator_.predict(X_test_scaled)

18750/18750 [==============================] - 14s 744us/step


In [11]:
# 저장
submission = pd.DataFrame({
    'ID': test['ID'],
    'Predicted_Segment': label_encoder.inverse_transform(y_test_pred)
})
submission.to_csv('merge/result/C2_XGb_예측.csv', index=False)